# Simple Python Project Demo: Jinja Templates for Prompts and Reports

This notebook demonstrates a small Python project using [Jinja](https://github.com/pallets/jinja/) / `jinja2` for templating.

The example is intentionally simple: we have a small JSON data file containing research-oriented extraction tasks, and we use external Jinja templates to generate:

1. task-specific LLM prompts, and  
2. a compact Markdown report.

The main point is to show why templating libraries are useful when prompt or document text contains conditional logic, loops, reusable variables, and externalized wording.


## 1. Install dependency

Run this cell if `jinja2` is not already installed in your notebook environment.


In [ ]:
# Uncomment if needed:
# %pip install jinja2

## 2. Create a tiny project structure

The notebook creates a mini project layout directly in the current working directory:

```text
jinja_notebook_demo/
├── data/
│   └── research_tasks.json
├── templates/
│   ├── report.md.j2
│   └── prompts/
│       └── task_prompt.txt.j2
└── outputs/
```


In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path("jinja_notebook_demo")
DATA_DIR = PROJECT_DIR / "data"
TEMPLATES_DIR = PROJECT_DIR / "templates"
PROMPTS_DIR = TEMPLATES_DIR / "prompts"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

for directory in [DATA_DIR, PROMPTS_DIR, OUTPUTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_DIR.resolve()


## 3. Create a small data file

The data file contains three fictional tasks. Each task has different properties so that the templates can demonstrate conditional logic.

For example:

- some tasks require citations,
- some have missing context files,
- different tasks request different output formats,
- high-priority tasks get additional quality checks.


In [ ]:
research_tasks = {
    "project": {
        "name": "Scientific Information Extraction Demo",
        "owner": "Example Research Group",
        "default_language": "English"
    },
    "tasks": [
        {
            "id": "T1",
            "title": "Extract materials and process parameters",
            "domain": "materials science",
            "input_file": "paper_ald_zno.pdf",
            "context_file": "ald_schema.json",
            "output_format": "json",
            "requires_citations": True,
            "priority": "high"
        },
        {
            "id": "T2",
            "title": "Summarize study design",
            "domain": "psychology",
            "input_file": "study_methods.xml",
            "context_file": None,
            "output_format": "markdown",
            "requires_citations": False,
            "priority": "medium"
        },
        {
            "id": "T3",
            "title": "Extract species-location observations",
            "domain": "invasion biology",
            "input_file": "ecology_paper.pdf",
            "context_file": "species_location_schema.json",
            "output_format": "csv",
            "requires_citations": True,
            "priority": "low"
        }
    ]
}

data_path = DATA_DIR / "research_tasks.json"
data_path.write_text(json.dumps(research_tasks, indent=2), encoding="utf-8")

print(f"Wrote data file to: {data_path}")


## 4. Create external Jinja templates

The key idea is that the prompt/report wording lives outside the Python rendering logic.

This is especially useful when prompts become longer and contain conditional instructions such as:

```jinja
{% if task.requires_citations %}
Include citations.
{% else %}
Citations are optional.
{% endif %}
```


In [ ]:
prompt_template = 'You are assisting with a scientific information extraction task.\n\nProject: {{ project.name }}\nTask ID: {{ task.id }}\nTask: {{ task.title }}\nDomain: {{ task.domain }}\nInput file: {{ task.input_file }}\n\n{% if task.context_file %}\nUse the following context/schema file when extracting information:\n- {{ task.context_file }}\n{% else %}\nNo external schema file is available. Infer a compact structure from the document, but clearly mark assumptions.\n{% endif %}\n\nExpected output format: {{ task.output_format | upper }}\n\n{% if task.output_format == "json" %}\nReturn only valid JSON. Do not include Markdown fences.\n{% elif task.output_format == "csv" %}\nReturn a CSV table with a header row.\n{% elif task.output_format == "markdown" %}\nReturn a concise Markdown summary with clear section headings.\n{% else %}\nReturn a clear structured response.\n{% endif %}\n\n{% if task.requires_citations %}\nEvery extracted claim should include a citation to the relevant source passage.\n{% else %}\nCitations are optional for this task.\n{% endif %}\n\n{% if task.priority == "high" %}\nThis is a high-priority task. Before finalizing, check for:\n1. missing required fields,\n2. inconsistent terminology,\n3. unsupported claims.\n{% endif %}'

report_template = '# {{ project.name }} — Task Overview\n\nOwner: {{ project.owner }}  \nDefault language: {{ project.default_language }}\n\n{% set high_priority_tasks = tasks | selectattr("priority", "equalto", "high") | list %}\nTotal tasks: {{ tasks | length }}  \nHigh-priority tasks: {{ high_priority_tasks | length }}\n\n## Tasks\n\n{% for task in tasks %}\n### {{ loop.index }}. {{ task.title }}\n\n- **Task ID:** {{ task.id }}\n- **Domain:** {{ task.domain }}\n- **Input file:** {{ task.input_file }}\n- **Output format:** {{ task.output_format }}\n- **Priority:** {{ task.priority }}\n\n{% if task.context_file %}\nSchema/context available: `{{ task.context_file }}`\n{% else %}\nNo schema/context file provided; template instructs the model to mark assumptions.\n{% endif %}\n\n{% if task.requires_citations %}\nCitation requirement: required.\n{% else %}\nCitation requirement: optional.\n{% endif %}\n\n{% endfor %}'

prompt_template_path = PROMPTS_DIR / "task_prompt.txt.j2"
report_template_path = TEMPLATES_DIR / "report.md.j2"

prompt_template_path.write_text(prompt_template, encoding="utf-8")
report_template_path.write_text(report_template, encoding="utf-8")

print(f"Wrote prompt template to: {prompt_template_path}")
print(f"Wrote report template to: {report_template_path}")


## 5. Render the templates

The Python code stays small because the conditional wording is handled by the templates.


In [ ]:
from jinja2 import Environment, FileSystemLoader, StrictUndefined

env = Environment(
    loader=FileSystemLoader(TEMPLATES_DIR),
    undefined=StrictUndefined,
    trim_blocks=True,
    lstrip_blocks=True,
)

with data_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

project = data["project"]
tasks = data["tasks"]

prompt_template = env.get_template("prompts/task_prompt.txt.j2")
report_template = env.get_template("report.md.j2")

# Render one prompt per task
for task in tasks:
    rendered_prompt = prompt_template.render(project=project, task=task)
    output_path = OUTPUTS_DIR / f"prompt_{task['id'].lower()}.txt"
    output_path.write_text(rendered_prompt, encoding="utf-8")

# Render one project report
rendered_report = report_template.render(project=project, tasks=tasks)
report_output_path = OUTPUTS_DIR / "task_report.md"
report_output_path.write_text(rendered_report, encoding="utf-8")

print("Generated files:")
for path in sorted(OUTPUTS_DIR.iterdir()):
    print("-", path)


## 6. Inspect one generated prompt

Notice how the generated prompt changes depending on the task metadata.


In [ ]:
print((OUTPUTS_DIR / "prompt_t1.txt").read_text(encoding="utf-8"))

## 7. Inspect another generated prompt

This second task has no context/schema file and does not require citations, so the generated instructions are different.


In [ ]:
print((OUTPUTS_DIR / "prompt_t2.txt").read_text(encoding="utf-8"))

## 8. Inspect the generated Markdown report

In [ ]:
from IPython.display import Markdown, display

display(Markdown((OUTPUTS_DIR / "task_report.md").read_text(encoding="utf-8")))


## 9. What this demonstrates

This notebook shows a simple but realistic pattern:

- keep prompt/report text in external `.j2` files,
- keep structured task metadata in a small data file,
- use Python only to load data and render templates,
- use Jinja conditionals and loops for task-specific wording.

This approach scales well when you need many prompt variants without hard-coding long strings inside Python scripts.
